# Predict

Cached studies in, `submission.csv` out. 
Read one step at a time: build any missing cache, load one checkpoint per fold, score under TTA, rank-mean the folds, write submission.

Point `COMP_DIR` at the competition mount and `WEIGHTS` at a directory holding `best_fold*.pt`.

In [ ]:
from pathlib import Path

from knee.cache import CacheConfig, build_cache, find_root, load_study_rows, pending_studies
from knee.predict import InferConfig, find_checkpoints, load_models, predict, write_submission

COMP_DIR = "data"
WEIGHTS = "weights/convnet_2p5d_exp019"
WORK_DIR = Path("/tmp/predict")
TTA = 2

cache_dir = WORK_DIR / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
cfg = InferConfig(comp_dir=COMP_DIR, out_dir=str(cache_dir), tta_offsets=tuple(range(TTA)))
cfg

## Cache

One `<study>.npz` per study. Already-cached studies are skipped, so this cell resumes.

In [ ]:
root = find_root(cfg.comp_dir, cfg.split)
study_rows = load_study_rows(root, cfg.split)
studies = sorted(study_rows)
todo = pending_studies(studies, cache_dir)
print(f"root: {root} | {len(studies)} studies | {len(todo)} to cache")

if todo:
    _, _, counts = build_cache(todo, study_rows, root / f"{cfg.split}_series", cache_dir,
                               CacheConfig(n_shards=1))
    print(counts)

## Models

All five folds of one run, or nothing: a partial set would quietly score worse.

In [ ]:
ckpts = find_checkpoints(cfg.n_folds, bases=[Path(WEIGHTS)])
models = load_models(ckpts, cfg)
print(f"{len(models)} models from {ckpts[0].parent} | device {cfg.device}")

## Submission

`write_submission` probes the cost on the first studies, trims TTA passes and then folds if the budget is tight, predicts, fills studies with no cache file with 0.5 and checks that no label column came out constant.

In [ ]:
sub = write_submission(models, studies, WORK_DIR / "submission.csv", cfg)
sub.head()

In [ ]:
# One study end to end, for a look at the raw per-fold probabilities behind a row.
raw = predict(models, studies[:1], list(cfg.tta_offsets), cfg)
print(raw.shape)
raw[:, 0].round(3)